# Data Transformation

## Objective

In this notebook, we will cover:

### Encoding Techniques
- One-Hot Encoding
- Label Encoding
- Target Encoding
- Frequency Encoding

### Numerical Transformation
- Standardization
- Min-Max Scaling
- Robust Scaling
- Log Transformation
- Square Root Transformation
- Polynomial Transformation

### Feature Engineering
- Polynomial Features
- Binning
- Interaction Features
- Date Feature Extraction
- Aggregation Features

The original Heart Failure dataset will remain unchanged.

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    PolynomialFeatures
)

In [3]:
df = pd.read_csv(
    "../heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex
0,75.0,0,582,0,20,1,265000.00,1.9,130,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0


In [4]:
df_transform = df.copy()

print("Shape:", df_transform.shape)
print("Missing Values:", df_transform.isnull().sum().sum())

Shape: (299, 10)
Missing Values: 0


# Encoding Techniques

Encoding converts categorical values into numerical representations that Machine Learning models can process.

In [5]:
encoding_data = pd.DataFrame({
    "city": [
        "Lahore",
        "Karachi",
        "Islamabad",
        "Lahore",
        "Karachi",
        "Lahore"
    ],
    "target": [
        1, 0, 1, 1, 0, 0
    ]
})

encoding_data

,city,target
0,Lahore,1
1,Karachi,0
2,Islamabad,1
3,Lahore,1
4,Karachi,0
5,Lahore,0


In [6]:
one_hot_encoded = pd.get_dummies(
    encoding_data,
    columns=["city"],
    dtype=int
)

one_hot_encoded


,target,city_Islamabad,city_Karachi,city_Lahore
0,1,0,0,1
1,0,0,1,0
2,1,1,0,0
3,1,0,0,1
4,0,0,1,0
5,0,0,0,1


In [7]:
label_data = encoding_data.copy()

label_encoder = LabelEncoder()

label_data["city_encoded"] = (
    label_encoder.fit_transform(
        label_data["city"]
    )
)

label_data

,city,target,city_encoded
0,Lahore,1,2
1,Karachi,0,1
2,Islamabad,1,0
3,Lahore,1,2
4,Karachi,0,1
5,Lahore,0,2


In [8]:
label_mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(
            label_encoder.classes_
        )
    )
)

label_mapping

{'Islamabad': np.int64(0), 'Karachi': np.int64(1), 'Lahore': np.int64(2)}

## Target Encoding

Target Encoding replaces each category with the average target value for that category.

This technique must be used carefully because it can cause data leakage.

In [9]:
target_data = encoding_data.copy()

target_means = target_data.groupby(
    "city"
)["target"].mean()

target_data["city_target_encoded"] = (
    target_data["city"].map(
        target_means
    )
)

target_data

,city,target,city_target_encoded
0,Lahore,1,0.666667
1,Karachi,0,0.000000
2,Islamabad,1,1.000000
3,Lahore,1,0.666667
4,Karachi,0,0.000000
5,Lahore,0,0.666667


In [10]:
target_means

city
Islamabad    1.000000
Karachi      0.000000
Lahore       0.666667
Name: target, dtype: float64

In [11]:
frequency_data = encoding_data.copy()

city_frequency = (
    frequency_data["city"]
    .value_counts()
)

frequency_data["city_frequency"] = (
    frequency_data["city"]
    .map(city_frequency)
)

frequency_data

,city,target,city_frequency
0,Lahore,1,3
1,Karachi,0,2
2,Islamabad,1,1
3,Lahore,1,3
4,Karachi,0,2
5,Lahore,0,3


# Numerical Transformation

Standard, Min-Max, Robust, and Log transformations were already covered in the previous notebook.

Here they are demonstrated briefly for comparison before moving to new transformations.

In [13]:
#Select Numerical Feature
feature = "serum_creatinine"

numerical_data = df[[feature]].copy()

numerical_data.head()

,serum_creatinine
0,1.9
1,1.1
2,1.3
3,1.9
4,2.7


In [14]:
#Compare Scaling Methods
standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

numerical_comparison = pd.DataFrame({
    "Original": df[feature],

    "Standard": standard_scaler.fit_transform(
        df[[feature]]
    ).flatten(),

    "MinMax": minmax_scaler.fit_transform(
        df[[feature]]
    ).flatten(),

    "Robust": robust_scaler.fit_transform(
        df[[feature]]
    ).flatten(),

    "Log": np.log1p(
        df[feature]
    )
})

numerical_comparison.head(10)

,Original,Standard,MinMax,Robust,Log
0,1.9,0.490057,0.157303,1.6,1.064711
1,1.1,-0.284552,0.067416,0.0,0.741937
2,1.3,-0.090900,0.089888,0.4,0.832909
3,1.9,0.490057,0.157303,1.6,1.064711
4,2.7,1.264666,0.247191,3.2,1.308333
5,2.1,0.683709,0.179775,2.0,1.131402
6,1.2,-0.187726,0.078652,0.2,0.788457
7,1.1,-0.284552,0.067416,0.0,0.741937
8,1.5,0.102752,0.112360,0.8,0.916291
9,9.4,7.752020,1.000000,16.6,2.341806


## Square Root Transformation

Square Root Transformation can reduce moderate right-skewness in non-negative numerical data.

In [15]:
sqrt_feature = "creatinine_phosphokinase"

df_sqrt = df.copy()

df_sqrt[
    "creatinine_phosphokinase_sqrt"
] = np.sqrt(
    df_sqrt[sqrt_feature]
)

df_sqrt[
    [
        sqrt_feature,
        "creatinine_phosphokinase_sqrt"
    ]
].head(10)

,creatinine_phosphokinase,creatinine_phosphokinase_sqrt
0,582,24.124676
1,7861,88.662281
2,146,12.083046
3,111,10.535654
4,160,12.649111
5,47,6.855655
6,246,15.684387
7,315,17.748239
8,157,12.529964
9,123,11.090537


In [16]:
print(
    "Original Skewness:",
    round(
        df[sqrt_feature].skew(),
        2
    )
)

print(
    "Square Root Skewness:",
    round(
        df_sqrt[
            "creatinine_phosphokinase_sqrt"
        ].skew(),
        2
    )
)

Original Skewness: 4.46
Square Root Skewness: 2.11


## Polynomial Transformation

Polynomial Transformation creates higher-order versions of numerical features, such as X².

In [17]:
poly_single = PolynomialFeatures(
    degree=2,
    include_bias=False
)

poly_values = poly_single.fit_transform(
    df[["age"]]
)

poly_age = pd.DataFrame(
    poly_values,
    columns=poly_single.get_feature_names_out(
        ["age"]
    )
)

poly_age.head()

,age,age^2
0,75.0,5625.0
1,55.0,3025.0
2,65.0,4225.0
3,50.0,2500.0
4,65.0,4225.0


# Feature Engineering

Feature Engineering creates new useful variables from existing information.

In [18]:
poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

feature_pair = df[
    [
        "age",
        "serum_creatinine"
    ]
]

polynomial_features = poly.fit_transform(
    feature_pair
)

polynomial_df = pd.DataFrame(
    polynomial_features,
    columns=poly.get_feature_names_out(
        feature_pair.columns
    )
)

polynomial_df.head()

,age,serum_creatinine,age^2,age serum_creatinine,serum_creatinine^2
0,75.0,1.9,5625.0,142.5,3.61
1,55.0,1.1,3025.0,60.5,1.21
2,65.0,1.3,4225.0,84.5,1.69
3,50.0,1.9,2500.0,95.0,3.61
4,65.0,2.7,4225.0,175.5,7.29


## Binning

Binning converts continuous numerical values into meaningful groups.

In [19]:
df_binning = df.copy()

df_binning["age_group"] = pd.cut(
    df_binning["age"],
    bins=[
        0,
        50,
        65,
        float("inf")
    ],
    labels=[
        "Younger",
        "Middle Age",
        "Older"
    ]
)

df_binning[
    [
        "age",
        "age_group"
    ]
].head(15)

,age,age_group
0,75.0,Older
1,55.0,Middle Age
2,65.0,Middle Age
3,50.0,Younger
4,65.0,Middle Age
5,90.0,Older
6,75.0,Older
7,60.0,Middle Age
8,65.0,Middle Age
9,80.0,Older


In [20]:
df_binning[
    "age_group"
].value_counts()

age_group
Middle Age    136
Older          89
Younger        74
Name: count, dtype: int64

## Interaction Features

Interaction Features combine multiple variables to represent their joint effect.

In [21]:
df_interaction = df.copy()

df_interaction[
    "age_creatinine_interaction"
] = (
    df_interaction["age"]
    * df_interaction["serum_creatinine"]
)

df_interaction[
    [
        "age",
        "serum_creatinine",
        "age_creatinine_interaction"
    ]
].head()

,age,serum_creatinine,age_creatinine_interaction
0,75.0,1.9,142.5
1,55.0,1.1,60.5
2,65.0,1.3,84.5
3,50.0,1.9,95.0
4,65.0,2.7,175.5


## Date Feature Extraction

Date columns can be transformed into useful features such as year, month, day, and day of week.

In [22]:
date_data = pd.DataFrame({
    "visit_date": [
        "2026-01-15",
        "2026-03-22",
        "2026-06-10",
        "2026-08-09"
    ]
})

date_data["visit_date"] = pd.to_datetime(
    date_data["visit_date"]
)

date_data

,visit_date
0,2026-01-15
1,2026-03-22
2,2026-06-10
3,2026-08-09


In [23]:
date_data["year"] = (
    date_data["visit_date"].dt.year
)

date_data["month"] = (
    date_data["visit_date"].dt.month
)

date_data["day"] = (
    date_data["visit_date"].dt.day
)

date_data["day_of_week"] = (
    date_data["visit_date"].dt.day_name()
)

date_data

,visit_date,year,month,day,day_of_week
0,2026-01-15,2026,1,15,Thursday
1,2026-03-22,2026,3,22,Sunday
2,2026-06-10,2026,6,10,Wednesday
3,2026-08-09,2026,8,9,Sunday


## Aggregation Features

Aggregation Features summarize multiple records that belong to the same entity.

Examples include total, average, count, minimum, and maximum values.

In [24]:
aggregation_data = pd.DataFrame({
    "patient_id": [
        1, 1, 1,
        2, 2,
        3, 3, 3
    ],
    "visit_cost": [
        1000, 1500, 1200,
        2000, 1800,
        900, 1100, 1300
    ]
})

aggregation_data

,patient_id,visit_cost
0,1,1000
1,1,1500
2,1,1200
3,2,2000
4,2,1800
5,3,900
6,3,1100
7,3,1300


In [25]:
patient_summary = (
    aggregation_data
    .groupby("patient_id")
    ["visit_cost"]
    .agg([
        "count",
        "sum",
        "mean",
        "max"
    ])
    .reset_index()
)

patient_summary

,patient_id,count,sum,mean,max
0,1,3,3700,1233.333333,1500
1,2,2,3800,1900.000000,2000
2,3,3,3300,1100.000000,1300


In [26]:
feature_engineering_summary = pd.DataFrame({
    "Original Age": df["age"].head(),
    "Age Group": df_binning["age_group"].head(),
    "Interaction Feature": (
        df_interaction[
            "age_creatinine_interaction"
        ].head()
    ),
    "Age Squared": (
        polynomial_df["age^2"].head()
    )
})

feature_engineering_summary

,Original Age,Age Group,Interaction Feature,Age Squared
0,75.0,Older,142.5,5625.0
1,55.0,Middle Age,60.5,3025.0
2,65.0,Middle Age,84.5,4225.0
3,50.0,Younger,95.0,2500.0
4,65.0,Middle Age,175.5,4225.0


In [27]:
print("Original Shape:", df.shape)

print(
    "Original Missing Values:",
    df.isnull().sum().sum()
)

print(
    "Original Columns:",
    df.columns.tolist()
)

Original Shape: (299, 10)
Original Missing Values: 0
Original Columns: ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex']


# Summary

In this notebook, we covered:

## Encoding Techniques

- One-Hot Encoding
- Label Encoding
- Target Encoding
- Frequency Encoding

## Numerical Transformation

- Standardization
- Min-Max Scaling
- Robust Scaling
- Log Transformation
- Square Root Transformation
- Polynomial Transformation

## Feature Engineering

- Polynomial Features
- Binning
- Interaction Features
- Date Feature Extraction
- Aggregation Features

## Key Learnings

- Encoding converts categorical variables into numerical representations.
- One-Hot Encoding is useful for unordered categories.
- Label Encoding assigns numerical labels.
- Target Encoding uses target information and has leakage risk.
- Frequency Encoding uses category occurrence counts.
- Square Root Transformation can reduce moderate skewness.
- Polynomial Transformation can represent nonlinear relationships.
- Binning creates meaningful groups.
- Interaction Features capture combined effects.
- Date columns can generate multiple useful features.
- Aggregation summarizes repeated observations.
- Engineered features should be validated before model training.